In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "3d5032c9",
   "metadata": {
    "vscode": {
     "languageId": "plaintext"
    }
   },
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.preprocessing import MinMaxScaler\n",
    "from sklearn.metrics import mean_absolute_error, mean_squared_error\n",
    "from statsmodels.tsa.arima.model import ARIMA\n",
    "from statsmodels.tsa.stattools import adfuller\n",
    "from statsmodels.graphics.tsaplots import plot_acf, plot_pacf\n",
    "from pmdarima import auto_arima\n",
    "from tensorflow.keras.models import Sequential\n",
    "from tensorflow.keras.layers import LSTM, Dense, Dropout\n",
    "from tensorflow.keras.callbacks import EarlyStopping\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set style\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "\n",
    "print(\"Libraries imported successfully!\")\n",
    "\n",
    "\n",
    "# Load data\n",
    "closing_prices = pd.read_csv('../data/processed/closing_prices.csv', index_col=0, parse_dates=True)\n",
    "returns = pd.read_csv('../data/processed/daily_returns.csv', index_col=0, parse_dates=True)\n",
    "\n",
    "# Focus on TSLA\n",
    "tsla_prices = closing_prices['TSLA']\n",
    "tsla_returns = returns['TSLA']\n",
    "\n",
    "print(f\"TSLA Data Shape: {tsla_prices.shape}\")\n",
    "print(f\"Date Range: {tsla_prices.index[0]} to {tsla_prices.index[-1]}\")\n",
    "print(f\"\\nTSLA Price Statistics:\")\n",
    "print(tsla_prices.describe())\n",
    "\n",
    "# Plot TSLA prices\n",
    "fig, axes = plt.subplots(2, 1, figsize=(15, 10))\n",
    "\n",
    "# Price\n",
    "axes[0].plot(tsla_prices.index, tsla_prices, label='TSLA Price', linewidth=2, color='blue')\n",
    "axes[0].set_title('TSLA Closing Price', fontsize=14, fontweight='bold')\n",
    "axes[0].set_xlabel('Date')\n",
    "axes[0].set_ylabel('Price ($)')\n",
    "axes[0].legend()\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "# Returns\n",
    "axes[1].plot(tsla_returns.index, tsla_returns, label='TSLA Returns', linewidth=1, color='green')\n",
    "axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.5)\n",
    "axes[1].set_title('TSLA Daily Returns', fontsize=14, fontweight='bold')\n",
    "axes[1].set_xlabel('Date')\n",
    "axes[1].set_ylabel('Return')\n",
    "axes[1].legend()\n",
    "axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/processed/tsla_overview.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "# Split data chronologically\n",
    "train_size = int(len(tsla_prices) * 0.8)\n",
    "train_prices = tsla_prices[:train_size]\n",
    "test_prices = tsla_prices[train_size:]\n",
    "\n",
    "train_returns = tsla_returns[:train_size]\n",
    "test_returns = tsla_returns[train_size:]\n",
    "\n",
    "print(f\"Training period: {train_prices.index[0]} to {train_prices.index[-1]}\")\n",
    "print(f\"Testing period: {test_prices.index[0]} to {test_prices.index[-1]}\")\n",
    "print(f\"\\nTraining size: {len(train_prices)} days\")\n",
    "print(f\"Testing size: {len(test_prices)} days\")\n",
    "print(f\"Total: {len(train_prices) + len(test_prices)} days\")\n",
    "\n",
    "# Plot split\n",
    "fig, ax = plt.subplots(figsize=(15, 6))\n",
    "ax.plot(train_prices.index, train_prices, label='Training Data', linewidth=2, color='blue')\n",
    "ax.plot(test_prices.index, test_prices, label='Testing Data', linewidth=2, color='red')\n",
    "ax.axvline(x=train_prices.index[-1], color='gray', linestyle='--', linewidth=2, \n",
    "           label='Train/Test Split')\n",
    "ax.set_title('TSLA Price - Train/Test Split', fontsize=14, fontweight='bold')\n",
    "ax.set_xlabel('Date')\n",
    "ax.set_ylabel('Price ($)')\n",
    "ax.legend()\n",
    "ax.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/processed/train_test_split.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"SARIMA MODEL\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "# Auto_arima with seasonality\n",
    "print(\"Finding optimal SARIMA parameters...\")\n",
    "auto_seasonal = auto_arima(\n",
    "    train_prices,\n",
    "    start_p=0, max_p=5,\n",
    "    start_q=0, max_q=5,\n",
    "    start_d=0, max_d=2,\n",
    "    seasonal=True,\n",
    "    m=12,  # Monthly seasonality (12 months)\n",
    "    trace=True,\n",
    "    error_action='ignore',\n",
    "    suppress_warnings=True,\n",
    "    stepwise=True,\n",
    "    n_fits=50,\n",
    "    scoring='mse'\n",
    ")\n",
    "\n",
    "if auto_seasonal.seasonal_order != (0, 0, 0, 0):\n",
    "    print(f\"\\nBest SARIMA parameters: {auto_seasonal.order}, seasonal={auto_seasonal.seasonal_order}\")\n",
    "    \n",
    "    # Fit SARIMA\n",
    "    sarima_model = ARIMA(train_prices, \n",
    "                         order=auto_seasonal.order,\n",
    "                         seasonal_order=auto_seasonal.seasonal_order)\n",
    "    sarima_fitted = sarima_model.fit()\n",
    "    \n",
    "    print(\"\\nModel Summary:\")\n",
    "    print(sarima_fitted.summary())\n",
    "    \n",
    "    # Generate forecasts\n",
    "    sarima_forecast = sarima_fitted.forecast(steps=len(test_prices))\n",
    "    sarima_forecast_series = pd.Series(sarima_forecast, index=test_prices.index)\n",
    "    \n",
    "    # Calculate metrics\n",
    "    sarima_mae = mean_absolute_error(test_prices, sarima_forecast_series)\n",
    "    sarima_rmse = np.sqrt(mean_squared_error(test_prices, sarima_forecast_series))\n",
    "    sarima_mape = np.mean(np.abs((test_prices - sarima_forecast_series) / test_prices)) * 100\n",
    "    \n",
    "    print(f\"\\nSARIMA Performance Metrics:\")\n",
    "    print(f\"  MAE: ${sarima_mae:.2f}\")\n",
    "    print(f\"  RMSE: ${sarima_rmse:.2f}\")\n",
    "    print(f\"  MAPE: {sarima_mape:.2f}%\")\n",
    "    \n",
    "    # Plot SARIMA forecast\n",
    "    fig, ax = plt.subplots(figsize=(15, 6))\n",
    "    ax.plot(train_prices.index, train_prices, label='Training Data', linewidth=1, alpha=0.7)\n",
    "    ax.plot(test_prices.index, test_prices, label='Actual Test Data', linewidth=2, color='blue')\n",
    "    ax.plot(test_prices.index, sarima_forecast_series, label='SARIMA Forecast', \n",
    "            linewidth=2, linestyle='--', color='green')\n",
    "    ax.axvline(x=train_prices.index[-1], color='gray', linestyle='--', linewidth=2)\n",
    "    ax.set_title('SARIMA Model Forecast vs Actual', fontsize=14, fontweight='bold')\n",
    "    ax.set_xlabel('Date')\n",
    "    ax.set_ylabel('Price ($)')\n",
    "    ax.legend()\n",
    "    ax.grid(True, alpha=0.3)\n",
    "    plt.tight_layout()\n",
    "    plt.savefig('../data/processed/sarima_forecast.png', dpi=300, bbox_inches='tight')\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"No significant seasonality detected.\")\n",
    "    sarima_forecast_series = None\n",
    "\n",
    "    print(\"\\n\" + \"=\"*60)\n",
    "print(\"LSTM MODEL\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "# Prepare data for LSTM\n",
    "def create_sequences(data, window_size=60):\n",
    "    \"\"\"Create sequences for LSTM training\"\"\"\n",
    "    X, y = [], []\n",
    "    for i in range(window_size, len(data)):\n",
    "        X.append(data[i-window_size:i])\n",
    "        y.append(data[i])\n",
    "    return np.array(X), np.array(y)\n",
    "\n",
    "# Scale data\n",
    "scaler = MinMaxScaler()\n",
    "scaled_prices = scaler.fit_transform(tsla_prices.values.reshape(-1, 1))\n",
    "\n",
    "# Split scaled data\n",
    "train_scaled = scaled_prices[:train_size]\n",
    "test_scaled = scaled_prices[train_size:]\n"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}